# 🦜🔍 LangSmith — Complete  Guide
### Tracing · Prompt Version Control · Golden Datasets · Evaluation
---
> **Models used:** `gpt-4o-mini` (chat) · `text-embedding-3-small` (embeddings)  
> **Packages:** `langsmith==0.8.x` · `langchain==1.3.x` · `langchain-openai==1.2.x`  
> **Secrets loaded from:** Google Colab Secret Manager (🔑 left panel)

---

## 📚 Table of Contents
1. [Setup & Installation](#setup)
2. [Configure Secrets (Colab)](#secrets)
3. [Module 1 — LangSmith Tracing Fundamentals](#tracing)
4. [Module 2 — @traceable Decorator & Custom Spans](#traceable)



---
## 1. 🛠️ Setup & Installation <a name="setup"></a>
Install all required packages. This cell only needs to run once per Colab session.


In [3]:
# ── Install all dependencies ──────────────────────────────────────────────────
!pip install -q \
    langsmith>=0.8.0 \
    langchain>=1.3.0 \
    langchain-openai>=1.2.0 \
    langchain-core>=0.3.0 \
    langchain-text-splitters>=0.3.0 \
    langchain-community>=0.4.0 \
    langchain-chroma>=0.2.0 \
    chromadb \
    faiss-cpu \
    tiktoken \
    openai

print("✅ All packages installed")


✅ All packages installed


---
## 2. 🔑 Configure Secrets (Colab Secret Manager) <a name="secrets"></a>

**Before running this cell:**
1. Click the 🔑 **key icon** in the left sidebar
2. Add the following secrets:
   - `OPENAI_API_KEY` — your OpenAI key
   - `LANGCHAIN_API_KEY` — your LangSmith API key (from smith.langchain.com → Settings → API Keys)
   - `LANGCHAIN_PROJECT` — a project name like `llmops-day14` (will be created if it doesn't exist)

> **Note:** `LANGCHAIN_TRACING_V2=true` enables automatic tracing for all LangChain chains.


In [4]:
import os
from google.colab import userdata

# ── Load secrets from Colab Secret Manager ────────────────────────────────────
os.environ["OPENAI_API_KEY"]        = userdata.get("OPENAI_API_KEY")
os.environ["LANGCHAIN_API_KEY"]     = userdata.get("LANGCHAIN_API_KEY")
os.environ["LANGCHAIN_TRACING_V2"]  = "true"          # Enable auto-tracing
os.environ["LANGCHAIN_PROJECT"]     = "Day 5 Tracing"
os.environ["LANGCHAIN_ENDPOINT"]    = "https://api.smith.langchain.com"

# ── Verify connection ──────────────────────────────────────────────────────────
from langsmith import Client
client = Client()

# Test connection
projects = list(client.list_projects())
print(f"✅ Connected to LangSmith")
print(f"📂 Available projects: {[p.name for p in projects[:5]]}")
print(f"🎯 Active project: {os.environ['LANGCHAIN_PROJECT']}")


✅ Connected to LangSmith
📂 Available projects: ['llmops-day14', 'rag-pipeline-v1-96a03715', 'support-bot-v1-comparison-d63e7707', 'support-bot-v3-defe7107', 'langchain-complete-guide']
🎯 Active project: Day 5 Tracing


---
## 3. 🔭 Module 1 — LangSmith Tracing Fundamentals <a name="tracing"></a>

### What Is Tracing?
LangSmith tracing records every step of your LLM pipeline:
- **Runs** = individual LLM calls, chain steps, tool calls
- **Traces** = a full tree of runs triggered by one user request
- **Spans** = named sections of work (auto-created for LangChain components)

### How It Works
Setting `LANGCHAIN_TRACING_V2=true` **automatically instruments** all LangChain components — no code changes needed. Every `chain.invoke()`, `llm.invoke()`, and `retriever.get_relevant_documents()` generates a trace.


In [5]:
# ── Imports ────────────────────────────────────────────────────────────────────
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# ── Models ─────────────────────────────────────────────────────────────────────
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.3,
    # LangSmith tracing is automatic — no extra config needed here
)

embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    dimensions=1536  # Default; can reduce to 512 or 256 for efficiency
)

print(f"✅ LLM: {llm.model_name}")
print(f"✅ Embeddings: {embeddings.model}")


✅ LLM: gpt-4o-mini
✅ Embeddings: text-embedding-3-small


In [6]:
# ── Simple traced chain ────────────────────────────────────────────────────────
# This chain is AUTOMATICALLY traced because LANGCHAIN_TRACING_V2=true

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an LLMOps expert. Be concise and practical."),
    ("human", "{question}")
])

chain = prompt | llm | StrOutputParser()

# Every invocation creates a trace in LangSmith
response = chain.invoke({
    "question": "What is the difference between LangSmith and Langfuse?"
})

print("Response:", response)
print()
print("🔍 View this trace at: https://smith.langchain.com")
print(f"📁 Project: {os.environ['LANGCHAIN_PROJECT']}")


Response: LangSmith and Langfuse are both tools designed to enhance the development and deployment of language models, but they serve different purposes:

1. **LangSmith**: Primarily focuses on providing a framework for building, testing, and iterating on language models. It emphasizes model training, evaluation, and integration into applications, allowing developers to streamline their workflows.

2. **Langfuse**: Concentrates on monitoring and analyzing the performance of language models in production. It offers insights into user interactions, model behavior, and performance metrics, helping teams optimize and troubleshoot their deployed models.

In summary, LangSmith is more about model development, while Langfuse is geared towards monitoring and analytics post-deployment.

🔍 View this trace at: https://smith.langchain.com
📁 Project: Day 5 Tracing


In [7]:
# ── Adding metadata to traces ──────────────────────────────────────────────────
# Use RunnableConfig to attach metadata, tags, and run names to traces

response2 = chain.invoke(
    {"question": "Explain semantic caching in 2 sentences"},
    config={
        "run_name": "semantic-caching-explanation",   # Human-readable name in UI
        "tags": ["Day 5", "Boeing", "Demo"],       # Filter by these in UI
        "metadata": {                                   # Custom key-value pairs
            "student_cohort": "Batch 1",
            "module": "Tracing",
            "difficulty": "Basic",
        }
    }
)

print("Response:", response2)
print()
print("✅ Trace created with metadata tags — filter by 'day-14' in LangSmith UI")


Response: Semantic caching is a technique that stores the results of previously executed queries along with their semantic context, allowing for faster retrieval of relevant data in future queries. By leveraging the relationships and meanings of data, it reduces the need for redundant computations and improves overall query performance.

✅ Trace created with metadata tags — filter by 'day-14' in LangSmith UI


In [8]:
# ── Batch tracing — run multiple inputs and see all traces ────────────────────
questions = [
    "What is vLLM used for?",
    "Explain the KV cache in one sentence",
    "What is the difference between Canary and Blue-Green deployment?",
    "When would you use DVC over MLflow?",
]

print("Running batch of questions...")
results = chain.batch(
    [{"question": q} for q in questions],
    config={"tags": ["batch-demo", "day-14"]}
)

for q, r in zip(questions, results):
    print(f"Q: {q}")
    print(f"A: {r[:100]}...")
    print()

print("✅ All 4 traces visible in LangSmith under project:", os.environ["LANGCHAIN_PROJECT"])


Running batch of questions...
Q: What is vLLM used for?
A: vLLM is used for efficient inference of large language models (LLMs). It optimizes memory usage and ...

Q: Explain the KV cache in one sentence
A: The KV cache (key-value cache) stores the key-value pairs of previously computed attention outputs i...

Q: What is the difference between Canary and Blue-Green deployment?
A: Canary and Blue-Green deployments are both strategies for releasing software updates, but they diffe...

Q: When would you use DVC over MLflow?
A: Use DVC (Data Version Control) over MLflow when:

1. **Data Management**: You need robust data versi...

✅ All 4 traces visible in LangSmith under project: Day 5 Tracing


---
## 4. 🎯 Module 2 — @traceable Decorator & Custom Spans <a name="traceable"></a>

### Why Use @traceable?
`@traceable` lets you trace **any Python function** — not just LangChain components.
Use it to trace:
- Pre/post-processing steps
- Custom retrieval logic
- Business logic around your LLM calls
- Any function you want to see in the trace tree


In [9]:
from langsmith import traceable, Client
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
import time

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.3)

# ── @traceable on a simple function ───────────────────────────────────────────
@traceable(name="preprocess-user-query", tags=["preprocessing"])
def preprocess_query(raw_query: str) -> str:
    """Cleans and normalises the user query before sending to LLM."""
    cleaned = raw_query.strip().lower()
    # Remove filler words
    for word in ["please", "can you", "could you", "tell me"]:
        cleaned = cleaned.replace(word, "")
    return cleaned.strip()

# ── @traceable on a retrieval function ────────────────────────────────────────
@traceable(name="custom-retrieval", run_type="retriever")
def retrieve_context(query: str, documents: list[dict]) -> list[str]:
    """Simple keyword-based retrieval — traced as a 'retriever' span."""
    query_words = set(query.lower().split())
    scored = []
    for doc in documents:
        content_words = set(doc["content"].lower().split())
        score = len(query_words & content_words) / len(query_words)
        if score > 0:
            scored.append((score, doc["content"]))

    scored.sort(reverse=True)
    return [content for _, content in scored[:3]]

# ── @traceable on the full pipeline ───────────────────────────────────────────
@traceable(name="full-qa-pipeline", tags=["pipeline", "day-14"])
def run_qa_pipeline(user_query: str, knowledge_base: list[dict]) -> dict:
    """Full QA pipeline — each step appears as a child span in LangSmith."""
    start = time.time()

    # Step 1: Preprocess (traced as child span)
    clean_query = preprocess_query(user_query)

    # Step 2: Retrieve (traced as child span)
    context_docs = retrieve_context(clean_query, knowledge_base)
    context_str   = "\n".join(context_docs)

    # Step 3: Generate answer with LangChain (auto-traced)
    prompt = ChatPromptTemplate.from_messages([
        ("system", "Answer based only on the provided context. Be concise.\n\nContext:\n{context}"),
        ("human", "{question}")
    ])
    chain  = prompt | llm | StrOutputParser()
    answer = chain.invoke({"context": context_str, "question": clean_query})

    return {
        "query": user_query,
        "cleaned_query": clean_query,
        "num_docs_retrieved": len(context_docs),
        "answer": answer,
        "latency_ms": round((time.time() - start) * 1000, 2)
    }

# ── Sample knowledge base ──────────────────────────────────────────────────────
KB = [
    {"id": 1, "content": "vLLM is a high-throughput inference engine using PagedAttention for memory management."},
    {"id": 2, "content": "LangSmith provides tracing, datasets, and evaluation for LLM applications."},
    {"id": 3, "content": "Semantic caching stores LLM responses by meaning, not exact text, reducing API calls."},
    {"id": 4, "content": "MLflow tracks LLM experiments: prompts, parameters, metrics and artifacts."},
    {"id": 5, "content": "Canary releases gradually route traffic to new model versions to reduce risk."},
    {"id": 6, "content": "The KV cache in transformers stores key-value pairs to avoid recomputation."},
]

result = run_qa_pipeline("What tools can I use for LLM tracing?", KB)
print("Answer:", result["answer"])
print("Latency:", result["latency_ms"], "ms")
print("\n✅ Check LangSmith — you'll see a trace tree with 3 child spans")


Answer: You can use LangSmith for LLM tracing.
Latency: 784.3 ms

✅ Check LangSmith — you'll see a trace tree with 3 child spans


In [ ]:
# ── @traceable with metadata from run context ──────────────────────────────────

from langsmith.run_helpers import traceable, get_current_run_tree

@traceable(name="contextual-generation", run_type="llm")
def generate_with_context(prompt_text: str, user_id: str, feature: str) -> str:

    run_tree = get_current_run_tree()

    if run_tree:
        # Update existing metadata instead of assigning
        run_tree.metadata.update({
            "user_id": user_id,
            "feature": feature,
            "model": "gpt-4o-mini"
        })

    response = llm.invoke(prompt_text)
    return response.content


result = generate_with_context(
    "List 3 key benefits of using LangSmith in production",
    user_id="student-001",
    feature="llmops-module"
)

print(result)
print("\n✅ user_id and feature metadata attached to this trace")

LangSmith offers several benefits for production environments, particularly in the context of developing and deploying language models and applications. Here are three key benefits:

1. **Streamlined Model Management**: LangSmith provides tools for managing multiple versions of language models, making it easier to track changes, roll back to previous versions, and ensure that the most effective models are in use. This helps teams maintain consistency and reliability in their applications.

2. **Enhanced Collaboration**: The platform facilitates collaboration among team members by providing a centralized space for sharing insights, annotations, and feedback on language models. This collaborative environment can lead to improved model performance as diverse perspectives and expertise are integrated into the development process.

3. **Performance Monitoring and Analytics**: LangSmith offers robust analytics and monitoring tools that allow teams to evaluate model performance in real-time. 

Full RAG Pipeline with All LangSmith Features <a name="rag-pipeline"></a>



In [10]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langsmith import traceable, Client
import uuid

llm        = ChatOpenAI(model="gpt-4o-mini", temperature=0.3)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
client     = Client()

# ── Build a sample knowledge base ─────────────────────────────────────────────
LLMOPS_DOCS = [
    Document(page_content="""
LangSmith is an observability and evaluation platform for LLM applications built by LangChain.
It provides distributed tracing for LangChain pipelines, a prompt version control system
called the Hub, dataset management for golden datasets, and an evaluation framework to score
LLM outputs. LangSmith integrates with LangChain automatically when LANGCHAIN_TRACING_V2=true.
Every chain invocation, LLM call, and retrieval operation is captured as a trace.""",
        metadata={"source": "langsmith-overview", "topic": "tracing"}),

    Document(page_content="""
Langfuse is an open-source LLM observability platform that provides tracing, evaluation,
and prompt management. Unlike LangSmith which focuses on LangChain, Langfuse works with
any LLM framework through its SDK, OpenTelemetry integration, or direct API. Langfuse 4.x
uses the @observe decorator for tracing and the CallbackHandler for LangChain integration.
It stores traces, spans, and generations with full metadata.""",
        metadata={"source": "langfuse-overview", "topic": "tracing"}),

    Document(page_content="""
MLflow is an open-source platform for managing the ML lifecycle. For LLMs, it tracks
experiments including prompt versions, model configurations, evaluation metrics, and
inference artifacts. MLflow stores runs in an experiment, and you can compare multiple
runs side-by-side. The model registry allows you to promote a run to 'production' status.""",
        metadata={"source": "mlflow-overview", "topic": "experiment-tracking"}),

    Document(page_content="""
vLLM is a high-throughput inference engine for open-source LLMs. It implements
PagedAttention, which manages the KV cache memory like a virtual memory system,
enabling continuous batching and significantly higher throughput than naive serving.
vLLM exposes an OpenAI-compatible REST API, so existing code works without modification.""",
        metadata={"source": "vllm-overview", "topic": "deployment"}),

    Document(page_content="""
Semantic caching for LLMs stores model responses indexed by the semantic meaning of
the request, not the exact string. When a new request arrives, its embedding is compared
against cached request embeddings. If the cosine similarity exceeds a threshold (typically
0.90-0.95), the cached response is returned without calling the LLM. This can reduce
API costs by 30-60% for applications with repeated or similar queries.""",
        metadata={"source": "caching-overview", "topic": "optimization"}),

    Document(page_content="""
LiteLLM is a proxy and SDK that provides a unified OpenAI-compatible interface to 100+
LLM providers including OpenAI, Anthropic, Cohere, and self-hosted models. It supports
intelligent routing based on cost or latency, automatic fallbacks when a provider fails,
load balancing across multiple API keys, and per-request cost tracking.""",
        metadata={"source": "litellm-overview", "topic": "cost-optimization"}),
]

# Split documents
splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=50)
chunks   = splitter.split_documents(LLMOPS_DOCS)

# Build vector store
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="llmops-kb"
)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

print(f"✅ Knowledge base built: {len(chunks)} chunks from {len(LLMOPS_DOCS)} documents")


✅ Knowledge base built: 9 chunks from 6 documents


In [13]:
# ── Full traced RAG chain ──────────────────────────────────────────────────────

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an LLMOps expert. Be concise and practical."),
    ("human", "{question}")
])

def format_docs(docs):
    """Format retrieved documents into a single context string."""
    return "\n\n---\n\n".join([
        f"Source: {doc.metadata.get('source', 'unknown')}\n{doc.page_content}"
        for doc in docs
    ])

# Build the RAG chain using LCEL (LangChain Expression Language)
# Each step is automatically traced
rag_chain = (
    {
        "context":  retriever | RunnableLambda(format_docs),
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

# Test queries
test_queries = [
    "What is LangSmith and how does tracing work?",
    "How does semantic caching reduce costs?",
    "What is the difference between LangSmith and Langfuse?",
    "How does vLLM handle memory management?",
]

print("Testing RAG chain with tracing...\n")
for query in test_queries:
    response = rag_chain.invoke(
        query,
        config={
            "run_name": "rag-qa",
            "tags":     ["rag", "llmops-kb"],
            "metadata": {"query_type": "knowledge-base", "model": "gpt-4o-mini"}
        }
    )
    print(f"Q: {query}")
    print(f"A: {response[:200]}...\n")

print("✅ All RAG traces visible in LangSmith with full retrieval context")


Testing RAG chain with tracing...

Q: What is LangSmith and how does tracing work?
A: LangSmith is a tool designed for enhancing the development and debugging of language models. It provides capabilities for tracking and analyzing the interactions between users and language models, ena...

Q: How does semantic caching reduce costs?
A: Semantic caching reduces costs by:

1. **Data Reusability**: It stores the results of previous queries, allowing reuse of these results for similar future queries, reducing the need for repeated compu...

Q: What is the difference between LangSmith and Langfuse?
A: LangSmith and Langfuse are both tools designed to enhance the development and deployment of language models, but they serve different purposes:

1. **LangSmith**: Primarily focuses on improving the de...

Q: How does vLLM handle memory management?
A: vLLM manages memory efficiently through a combination of techniques:

1. **Memory Pooling**: It uses a memory pool to allocate and deallocate memo